# Checking db

In [30]:
import sqlite3
from pathlib import Path


import pandas as pd

from racecast.config import DB_PATH, REPO_ROOT

QUERIES_DIR = REPO_ROOT / "data" / "queries"
QUERIES_DIR.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

def save(df: pd.DataFrame, name: str) -> Path:
    path = REPO_ROOT/"data"/"queries"/name
    df.to_csv(path_or_buf=path)
    return path

In [35]:
%%sql
SELECT d.driver_ref,
       d.first_name || ' ' || d.last_name                                  AS name,
       MIN(se.year)                                                        AS first_season,
       MAX(se.year)                                                        AS last_season,
       COUNT(*) FILTER (WHERE r.session_type = 'race')                     AS races,
       COUNT(*) FILTER (WHERE r.session_type = 'race' AND r.position = 1)  AS wins,
       COUNT(*) FILTER (WHERE r.session_type = 'race' AND r.position <= 3) AS podiums,
       COUNT(*) FILTER (WHERE r.session_type = 'qualifying' AND r.position = 1) AS poles,
       COUNT(*) FILTER (WHERE r.session_type = 'race' AND r.dnf = 1)       AS dnfs,
       ROUND(SUM(r.points)  FILTER (WHERE r.session_type = 'race'), 1)     AS points,
       ROUND(AVG(r.position) FILTER (WHERE r.session_type = 'race'), 2)    AS avg_finish,
       ROUND(AVG(r.grid_position) FILTER (WHERE r.session_type = 'race'), 2) AS avg_grid
FROM drivers d
JOIN results  r  ON r.driver_id  = d.driver_id
JOIN sessions s  ON s.session_id = r.session_id
JOIN events   e  ON e.event_id   = s.event_id
JOIN seasons se  ON se.season_id = e.season_id
WHERE se.year >= 1980
GROUP BY d.driver_id
ORDER BY points DESC;

,driver_ref,name,first_season,last_season,races,wins,podiums,poles,dnfs,points,avg_finish,avg_grid
0,hamilton,Lewis Hamilton,2007,2026,393,106,207,107,32,5126.5,5.20,4.66
1,max_verstappen,Max Verstappen,2015,2026,246,71,132,51,34,3416.5,5.63,4.90
2,vettel,Sebastian Vettel,2007,2022,300,53,122,57,38,3098.0,7.09,6.27
3,alonso,Fernando Alonso,2001,2026,441,32,106,23,80,2383.0,8.92,9.03
4,raikkonen,Kimi Räikkönen,2001,2021,352,21,103,19,67,1873.0,8.49,7.53
...,...,...,...,...,...,...,...,...,...,...,...,...
267,keegan,Rupert Keegan,1980,1982,7,0,0,0,2,0.0,14.86,20.71
268,henton,Brian Henton,1981,1982,15,0,0,0,7,0.0,12.40,19.33
269,brambilla,Vittorio Brambilla,1980,1980,2,0,0,0,2,0.0,20.50,20.50
270,depailler,Patrick Depailler,1980,1980,8,0,0,0,8,0.0,15.75,11.13


In [32]:
save(driver_2018_up, "drivers_2018_up_era")

PosixPath('/Users/jjbrychta/PythonLab/RaceCast/data/queries/drivers_2018_up_era')

In [33]:
%%sql
SELECT * FROM drivers where driver_ref is null

,driver_id,driver_ref,first_name,last_name,abbreviation,country_code
